In [ ]:
# ── SETUP: Create required directories ───────────────────────────────────────
import os
for d in ['data', 'outputs', 'models']:
    os.makedirs(d, exist_ok=True)
print("Directory structure ready.")
print()
print("Before running this notebook, ensure the following files are in the data/ folder:")
print("  1. PDE4B_Data.csv   — ChEMBL PDE4B bioactivity data (CHEMBL275)")
print("     Download: https://www.ebi.ac.uk/chembl/target/CHEMBL275")
print("     Select: IC50 data → Download as CSV")
print()
print("  2. LOTUS_DB.smi     — LOTUS natural product SMILES")
print("     Download: https://lotus.naturalproducts.net/download")
print("     Select: SMILES file (.smi format)")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Draw import IPythonConsole
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import AllChem, PandasTools
from rdkit.Chem import Draw
from rdkit.Chem import rdMolDescriptors
from rdkit import DataStructs
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
import shap


from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, KFold
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.metrics import (
    precision_score, recall_score, f1_score, 
    accuracy_score, roc_auc_score, confusion_matrix, 
    mean_squared_error, r2_score, mean_absolute_error)
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_csv('data/PDE4B_Data.csv')  # Place your ChEMBL download here 
df_new = df[['Molecule ChEMBL ID', 'Smiles', 'pChEMBL Value']]
df_final = df_new.dropna(ignore_index=True)

In [ ]:
df_final

In [ ]:
df_final.isna().sum()

In [ ]:
# Creating RDKit molecule instances
df_final['Molecule'] = df_final['Smiles'].apply(Chem.MolFromSmiles)

In [ ]:
strong_inhibitors_count = df_final[df_final['pChEMBL Value'] >= 7].shape[0]
strong_inhibitors_count

In [ ]:
try:
    # Assuming the file name you used previously
    final_data_df = pd.read_csv('data/PDE4B_Data.csv')  # Place your ChEMBL download here
except FileNotFoundError:
    print("Error: Please check the file path for your final curated dataset.")
    # If the file isn't available, you must load the data from where you finished cleaning it
    # For now, we assume the data is correct.
    
# --- Plotting the Distribution ---
plt.figure(figsize=(9, 5))

# Use the column name from your dataset
pchembl_column_name = 'pChEMBL Value' # **Verify this column name in your CSV**

sns.histplot(
    final_data_df[pchembl_column_name], 
    kde=True, 
    bins=20, 
    color='skyblue',
    edgecolor='black'
)

# Add the critical cutoff line to justify the classification
CUTOFF_VALUE = 7 

# FIX: Use 'r' prefix for raw string to handle LaTeX symbols correctly
plt.axvline(
    x=CUTOFF_VALUE, 
    color='red', 
    linestyle='--', 
    linewidth=2, 
    label=r'Classification Cutoff ($\text{pChEMBL} \geq ' + str(CUTOFF_VALUE) + r'$)'
)

# FIX: Also ensure the title and axis labels are robustly handled
plt.title(r'Distribution of $\text{pChEMBL}$ Activity Values for $\text{PDE4B}$ Inhibitors', fontsize=16)
plt.xlabel(r'$\text{pChEMBL}$ Value', fontsize=14)
plt.ylabel('Frequency (Count)', fontsize=14)
plt.legend()
plt.grid(axis='y', alpha=0.5)

# Save the figure for your report
FIGURE_FILE_NAME = 'Figure_1_pChEMBL_Distribution.png'
plt.savefig(FIGURE_FILE_NAME, dpi=300, bbox_inches='tight')
plt.close() # Use plt.close() instead of plt.show() if running in a script environment

print(f"Figure 1 has been saved as '{FIGURE_FILE_NAME}'.")

In [ ]:
# 1. Create potency class column
df_final['potency_class'] = df_final['pChEMBL Value'].apply(
    lambda x: 'Strong (≥7)' if x >= 7 else 'Weak (<7)'
)

# 2. Count class distribution
class_counts = df_final['potency_class'].value_counts()
print(class_counts)

# 3. Plot bar chart of class balance
plt.figure()
class_counts.plot(kind='bar')

plt.title('Class Distribution of PDE4 Inhibitors')
plt.xlabel('Potency Class')
plt.ylabel('Number of Compounds')
plt.xticks(rotation=0)

plt.show()

In [ ]:
df_final

In [ ]:
df_final = df_final.drop('potency_class', axis=1)

In [ ]:
df_final

In [ ]:
df_final.Molecule[15]

In [ ]:
df_final.Molecule.isna().sum()

In [ ]:
# Get the list of 217 RDKit descriptors
des_list = [x[0] for x in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(des_list)

# Calculating descriptors for all molecules
descriptors = [calculator.CalcDescriptors(mol) for mol in df_final['Molecule']]

# Creating descriptor DataFrame and concatenate
descriptors_df = pd.DataFrame(descriptors, columns=des_list)
df_with_descriptors = pd.concat([df_final, descriptors_df], axis=1)

print(f"Total Entries after cleaning: {df_with_descriptors.shape[0]}")
print(f"Initial Features (RDKit Descriptors): {len(des_list)}")

In [ ]:
df_final

In [ ]:
df_with_descriptors

In [ ]:
des_list

In [ ]:
# Define Targets and Initial Features (X)
OPTIMAL_THRESHOLD = 7
df_with_descriptors['Activity_Class'] = np.where(df_with_descriptors['pChEMBL Value'] >= OPTIMAL_THRESHOLD, 1, 0)
X = df_with_descriptors[des_list]
y_class = df_with_descriptors['Activity_Class']
y_reg = df_with_descriptors['pChEMBL Value']

In [ ]:
sel_var = VarianceThreshold(threshold=0.01)
X_var_df = X.loc[:, sel_var.fit(X).get_support()]
corr_matrix = X_var_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.90)]
X_corr_df = X_var_df.drop(columns=to_drop)

In [ ]:
# SelectKBest (K=20)
K_FEATURES = 20
selector = SelectKBest(score_func=mutual_info_classif, k=K_FEATURES)
selector.fit(X_corr_df, y_class)
X_final = X_corr_df.loc[:, selector.get_support()]
print(f"Final Feature Count: {X_final.shape[1]}")

In [ ]:
X_final

In [ ]:
# Scaling and Split
scaler = StandardScaler()
X_scaled_df = pd.DataFrame(scaler.fit_transform(X_final), columns=X_final.columns)
X_train, X_test, y_train_class, y_test_class, y_train_reg, y_test_reg = train_test_split(
    X_scaled_df, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

In [ ]:
X_scaled_df

In [ ]:
plt.figure(figsize=(14, 6))
sns.heatmap(X_final.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, linewidths=.5, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of the 20 Final Selected Descriptors')
plt.show()

In [ ]:
y_class.value_counts()

In [ ]:
classifiers = {
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'SVM_Linear': SVC(kernel='linear', probability=True, random_state=42),
    'LightGBM': LGBMClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}
param_grid_class = {
    'XGBoost': {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.01, 0.05, 0.1]},
    'RandomForest': {'n_estimators': [100, 200], 'max_depth': [10, 20], 'min_samples_leaf': [1, 2, 5]},
    'SVM_Linear': {'C': [0.1, 1, 10]},
    'LightGBM': {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.05, 0.1]},
    'Gradient Boosting': {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1]},
}

cv_class = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_models_class = {}
cv_train_scores = {}  # store mean CV F1 on training folds

for name, model in classifiers.items():
    grid_search = GridSearchCV(model, param_grid_class[name], scoring='f1', cv=cv_class,
                               n_jobs=-1, verbose=0, return_train_score=True)
    grid_search.fit(X_train, y_train_class)
    best_models_class[name] = grid_search.best_estimator_
    # Mean CV F1 on training set (best param combo)
    best_idx = grid_search.best_index_
    cv_train_scores[name] = grid_search.cv_results_['mean_train_score'][best_idx]

# Final Classification Metrics Comparison (held-out test set)
from sklearn.metrics import matthews_corrcoef
class_final_results = {}
for name, best_model in best_models_class.items():
    y_pred  = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc          = accuracy_score(y_test_class, y_pred)
    auc          = roc_auc_score(y_test_class, y_proba)
    precision_c1 = precision_score(y_test_class, y_pred, pos_label=1)
    recall_c1    = recall_score(y_test_class, y_pred, pos_label=1)
    f1_c1        = f1_score(y_test_class, y_pred, pos_label=1)
    mcc          = matthews_corrcoef(y_test_class, y_pred)

    class_final_results[name] = {
        'CV Train F1 (mean)' : cv_train_scores[name],
        'Test Accuracy'      : acc,
        'Test AUC-ROC'       : auc,
        'Test F1-Score (C1)' : f1_c1,
        'Test Precision (C1)': precision_c1,
        'Test Recall (C1)'   : recall_c1,
        'Test MCC'           : mcc
    }

class_results_df = pd.DataFrame(class_final_results).T.sort_values(
    by='Test F1-Score (C1)', ascending=False)
best_class_model_name = class_results_df.index[0]

print("\n--- Classification Model Comparison (CV Train vs External Test Set) ---")
print(class_results_df.to_string(float_format='%.4f'))
print(f"\n Best Classification Model: **{best_class_model_name}**")


In [ ]:
print("\n--- Detailed Classification Reports (External Test Set) ---")
for name, best_model in best_models_class.items():
    y_pred = best_model.predict(X_test)
    print(f"\n=======================================================")
    print(f"CLASSIFICATION REPORT for: {name}")
    print(f"=======================================================")
    print(classification_report(
        y_test_class, 
        y_pred, 
        target_names=['Weak Inhibitor (0)', 'Strong Inhibitor (1)'], 
        digits=4
    ))

In [ ]:
class_labels = ['Weak Inhibitor (0)', 'Strong Inhibitor (1)']

for name, best_model in best_models_class.items():
    plt.figure(figsize=(7, 4))
    y_pred = best_model.predict(X_test)
    cm = confusion_matrix(y_test_class, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, xticklabels=class_labels, yticklabels=class_labels)
    plt.title(f'Confusion Matrix: {name}', fontsize=16)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

In [ ]:
print(class_results_df.columns.tolist())

In [ ]:
# 1. Define all tree-based model keys from your classifier dictionary
tree_models_to_check = ['XGBoost', 'RandomForest', 'LightGBM', 'Gradient Boosting']

# 2. Filter the results DataFrame to include only the tree models that were actually trained
# We use .intersection() to safely handle cases where a model might not be in the index
available_tree_results = class_results_df.loc[
    class_results_df.index.intersection(tree_models_to_check)
]

best_tree_model_name = None
if not available_tree_results.empty:
    # 3. Find the name of the model with the highest F1-Score (C1) among the available tree models
    best_tree_model_name = available_tree_results['Test F1-Score (C1)'].idxmax()

# --- Feature Importance Plotting ---
if best_tree_model_name: # Check if a best tree model was found
    best_tree_model = best_models_class[best_tree_model_name]
    
    # Check if the model has feature_importances_ (e.g., if SVM was somehow selected, though the logic above prevents it)
    if hasattr(best_tree_model, 'feature_importances_'):
        feature_importance_df = pd.DataFrame({
            'Feature': X_train.columns.tolist(),
            'Importance': best_tree_model.feature_importances_
        }).sort_values(by='Importance', ascending=False)
        
        print(f"\nTop 10 Feature Importance Scores for {best_tree_model_name}:\n")
        print(feature_importance_df.head(10).to_string(index=False, float_format='%.4f'))
        
        # Plotting code remains the same
        plt.figure(figsize=(10, 7))
        sns.barplot(x='Importance', y='Feature', data=feature_importance_df, color='darkred')
        plt.title(f'Feature Importance: {best_tree_model_name} Classifier (K={K_FEATURES})')
        plt.xlabel('Importance Score')
        plt.ylabel('Descriptor')
        plt.tight_layout()
        plt.show()
    else:
        # This clause handles cases where the best overall model (not tree-based) was not checked above
        print(f"Skipping Feature Importance plot as the best model, {best_tree_model_name}, does not support feature importance.")
else:
    print("Skipping Feature Importance plot as no tree-based models were found in the results.")

In [ ]:
best_model

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, plot_type="bar")

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_test)

In [ ]:
# Get model predictions
y_pred = best_model.predict(X_test)

# Convert to numpy for indexing
y_test_np = np.array(y_test_class)

# True active correctly predicted
active_index = np.where((y_test_np == 1) & (y_pred == 1))[0][0]

# True inactive correctly predicted
inactive_index = np.where((y_test_np == 0) & (y_pred == 0))[0][0]

print("Active index:", active_index)
print("Inactive index:", inactive_index)

In [ ]:
from sklearn.metrics import roc_curve, auc
print("\n--- AUC-ROC Curve Comparison Plot ---")
plt.figure(figsize=(6, 5))
# Iterate through all five classification models
for name, best_model in best_models_class.items():
    # 1. Get the probability scores for the positive class (Strong Inhibitor = 1)
    # This requires 'probability=True' in the SVM setup, which you already have.
    y_proba = best_model.predict_proba(X_test)[:, 1]
    
    # 2. Calculate the ROC curve points
    fpr, tpr, thresholds = roc_curve(y_test_class, y_proba)
    
    # 3. Calculate the Area Under the Curve (AUC)
    roc_auc = auc(fpr, tpr)
    
    # 4. Plot the curve
    plt.plot(
        fpr, tpr, 
        label=f'{name} (AUC = {roc_auc:.4f})', 
        linewidth=2
    )

# Plot the diagonal line (random classifier)
plt.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.50)')

# Set titles and labels
plt.title('Receiver Operating Characteristic (ROC) Curve Comparison', fontsize=16)
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity/Recall)', fontsize=12)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# MCC is now computed inside the model comparison block above (Cell 25).
# Displaying the consolidated results table again for reference:
print("\n--- MCC Summary (from model comparison table) ---")
print(class_results_df[["Test MCC"]].to_string(float_format='%.4f'))

In [ ]:
print("\n" + "="*80)
print("--- Probability Distribution of Predictions (Best Classifier) ---")
print("="*80)

best_model = best_models_class[best_class_model_name]
# Get the probability scores for the positive class (Strong Inhibitor = 1)
y_proba = best_model.predict_proba(X_test)[:, 1] 

# Separate the probabilities based on the TRUE label (y_test_class)
strong_inhibitor_probabilities = y_proba[y_test_class == 1]
weak_inhibitor_probabilities = y_proba[y_test_class == 0]

plt.figure(figsize=(10, 6))

# Plot the distribution for the two true classes
sns.histplot(
    strong_inhibitor_probabilities, 
    bins=30, 
    color="darkgreen", 
    kde=True, 
    label="Strong Inhibitor (True Positives)",
    alpha=0.6,
    line_kws={'linewidth': 2}
)

sns.histplot(
    weak_inhibitor_probabilities, 
    bins=30, 
    color="lightcoral", 
    kde=True, 
    label="Weak Inhibitor (True Negatives)",
    alpha=0.6,
    line_kws={'linewidth': 2}
)

# Add the default classification threshold line
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=1.5, label='Decision Threshold (0.5)')


plt.title(f'Predicted Probability Distribution: {best_class_model_name}', fontsize=16)
plt.xlabel('Predicted Probability of being Strong Inhibitor (Class 1)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(title='True Class')
plt.grid(True, which='both', linestyle=':', linewidth=0.5)
plt.xlim(0, 1)

# You can adjust the file name as needed
plt.savefig(f'{best_class_model_name}_predicted_probabilities_distribution.png')
plt.show()

print(f"The predicted probabilities distribution plot has been generated and saved for {best_class_model_name}.")

In [ ]:
best_models_class

In [ ]:
best_class_model_name

In [ ]:
best_model

In [ ]:
X_train.columns.to_list

In [ ]:
import joblib

In [ ]:
joblib.dump({
    'model': best_model,
    'features': X_train.columns.tolist()
}, 'models/best_pde4_model_with_features.pkl')

### For Model Loading

In [ ]:
# data = joblib.load('xgb_pde4_model_with_features.pkl')
# loaded_model = data['model']
# feature_names = data['features']

## SCREENING "ACTIVES" USING THE MODEL

In [ ]:
# Load SMILES
smi_file = 'data/LOTUS_DB.smi'  # Download from https://lotus.naturalproducts.net
lotus_df = pd.read_csv(smi_file, sep='\t', header=None, names=['SMILES', 'Name'])
print(f"Total compounds: {len(lotus_df)}")

In [ ]:
lotus_df

In [ ]:
# lotus_df.to_excel('outputs/LOTUS_Data.xlsx')  # Optional: save raw LOTUS data

In [ ]:
lotus_df.isna().sum()

In [ ]:
# List of descriptor names used in training (20 selected features)
descriptor_features = X_train.columns.to_list()

# Create a mapping from descriptor names to RDKit functions
desc_func_map = {name: getattr(Descriptors, name) for name in descriptor_features}

# Function to compute descriptors dynamically
def compute_descriptors_fast(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {name: func(mol) for name, func in desc_func_map.items()}

# Apply to all compounds in LOTUS
print("Computing descriptors for LOTUS compounds... (this may take a few minutes)")
descriptor_list = [compute_descriptors_fast(smi) for smi in lotus_df['SMILES']]

# Convert to DataFrame and drop failed molecules
descriptors_df = pd.DataFrame(descriptor_list).dropna().reset_index(drop=True)
lotus_df = lotus_df.loc[descriptors_df.index].reset_index(drop=True)

# ── CRITICAL FIX: apply the SAME StandardScaler fitted on training data ───────
# The model was trained on scaled features; raw descriptors MUST be transformed
# before prediction to avoid data mismatch.
X_lotus_raw    = descriptors_df[descriptor_features]
X_lotus_scaled = pd.DataFrame(
    scaler.transform(X_lotus_raw),
    columns=descriptor_features
)
print(f"Descriptor computation complete. Valid LOTUS compounds: {len(X_lotus_scaled)}")


In [ ]:
# Predict probability of being a strong PDE4B inhibitor (class 1)
# Using X_lotus_scaled (StandardScaler-transformed) — NOT raw descriptors
y_prob = best_model.predict_proba(X_lotus_scaled)[:, 1]
lotus_df['Predicted_Prob']  = y_prob

# Predicted class: 1 = strong inhibitor (pIC50 >= 7), 0 = weak
lotus_df['Predicted_Class'] = (y_prob >= 0.5).astype(int)

# Sort by predicted probability (highest confidence first)
top_hits = lotus_df.sort_values('Predicted_Prob', ascending=False).reset_index(drop=True)

# Summary
actives   = top_hits[top_hits['Predicted_Class'] == 1]
inactives = top_hits[top_hits['Predicted_Class'] == 0]
print(f"Total LOTUS compounds screened : {len(lotus_df)}")
print(f"Predicted strong inhibitors    : {len(actives)}")
print(f"Predicted weak/inactive        : {len(inactives)}")

# Save full predictions
top_hits.to_csv('outputs/LOTUS_DB_PDE4B_predictions.csv', index=False)
print("Predictions saved to LOTUS_DB_PDE4B_predictions.csv")


In [ ]:
actives = top_hits[top_hits['Predicted_Class'] == 1]
inactives = top_hits[top_hits['Predicted_Class'] == 0]

In [ ]:
actives

In [ ]:
# ── Applicability Domain: PCA overlay of training set vs LOTUS hits ──────────
# Run AFTER screening so that X_lotus_scaled is available.
# Shows whether predicted actives occupy the same chemical space as the training data.
from sklearn.decomposition import PCA as _PCA

pca_ad = _PCA(n_components=2, random_state=42)
pca_ad.fit(X_scaled_df)   # fit on full scaled training+test data

train_proj = pca_ad.transform(X_scaled_df)

# Project LOTUS actives (use X_lotus_scaled filtered to predicted actives)
lotus_actives_mask = (y_prob >= 0.5)  # boolean mask on X_lotus_scaled
lotus_proj = pca_ad.transform(X_lotus_scaled[lotus_actives_mask])

plt.figure(figsize=(8, 6))
plt.scatter(train_proj[:, 0], train_proj[:, 1],
            alpha=0.4, s=15, label='Training set', color='steelblue')
plt.scatter(lotus_proj[:, 0], lotus_proj[:, 1],
            alpha=0.6, s=15, label='LOTUS predicted actives', color='darkorange')
plt.xlabel(f'PC1 ({pca_ad.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca_ad.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Applicability Domain: PCA Chemical Space Coverage')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/Figure_AD_PCA_Coverage.png', dpi=300, bbox_inches='tight')
plt.show()
print("PCA applicability domain plot saved.")

## Further Screening

In [ ]:
actives_df1 = actives[['Name', 'SMILES']]

In [ ]:
actives_df1

In [ ]:
actives_df1.isna().sum()

In [ ]:
actives_df1['Molecule'] = actives_df1['SMILES'].apply(Chem.MolFromSmiles)

In [ ]:
actives_df1

In [ ]:
actives_df1.isna().sum()

In [ ]:
actives_df1.Molecule[2]

In [ ]:
from rdkit.Chem import Descriptors
from rdkit.Chem.Descriptors import rdMolDescriptors

In [ ]:
actives_df1['MolWt']           = actives_df1['Molecule'].apply(Descriptors.MolWt)
actives_df1['LogP']            = actives_df1['Molecule'].apply(Descriptors.MolLogP)
actives_df1['HbondDonors']     = actives_df1['Molecule'].apply(Descriptors.NumHDonors)
actives_df1['HbondAcceptors']  = actives_df1['Molecule'].apply(Descriptors.NumHAcceptors)
actives_df1['RotatableBonds']  = actives_df1['Molecule'].apply(Descriptors.NumRotatableBonds)

print(actives_df1[['MolWt','LogP','HbondDonors','HbondAcceptors','RotatableBonds']].describe())


In [ ]:
actives_df1

In [ ]:
# Lipinski Rule-of-Five filter (zero violations tolerated)
# Ro5: MW ≤ 500, LogP ≤ 5, HBD ≤ 5, HBA ≤ 10, RotBonds ≤ 10
pre_filter_count = len(actives_df1)

df_actives_filtered = actives_df1[
    (actives_df1['MolWt']          <= 500) &
    (actives_df1['LogP']           <= 5)   &
    (actives_df1['HbondDonors']    <= 5)   &
    (actives_df1['HbondAcceptors'] <= 10)  &
    (actives_df1['RotatableBonds'] <= 10)   # Lipinski Ro5 rotatable bonds criterion
].copy()

print(f"Compounds before Lipinski filter : {pre_filter_count}")
print(f"Compounds after  Lipinski filter : {len(df_actives_filtered)}")
print(f"Removed by Lipinski filter       : {pre_filter_count - len(df_actives_filtered)}")


In [ ]:
df_actives_filtered

In [ ]:
df_actives_filtered['MolWt'].hist(figsize=(9, 6), grid=False)

In [ ]:
df_actives_filtered['LogP'].hist(figsize=(9, 6), grid=False)

In [ ]:
df_actives_filtered['HbondDonors'].hist(figsize=(9, 6), grid=False)

In [ ]:
df_actives_filtered['HbondAcceptors'].hist(figsize=(9, 6), grid=False)

In [ ]:
df_actives_filtered['RotatableBonds'].hist(figsize=(9, 6), grid=False)
plt.title('Distribution of Rotatable Bonds (Lipinski Ro5 Filter)')
plt.xlabel('Number of Rotatable Bonds')
plt.ylabel('Count')
# plt.axvline(x=10, color='red', linestyle='--', label='Ro5 cutoff (≤10)')
plt.legend()
plt.show()

### Screening by QED

In [ ]:
df_actives_filtered['QED'] = df_actives_filtered['Molecule'].apply(Descriptors.qed)

In [ ]:
df_actives_filtered

In [ ]:
df_actives_filtered['QED'].hist(figsize=(9, 6), grid=False)

In [ ]:
QED_THRESHOLD = 0.70   # lowered from 0.85; aligns better with natural product chemical space
pre_qed_count = len(df_actives_filtered)

df_actives_filtered_qed = df_actives_filtered[
    df_actives_filtered['QED'] >= QED_THRESHOLD
].copy()

print(f"Compounds before QED filter (>= {QED_THRESHOLD}): {pre_qed_count}")
print(f"Compounds after  QED filter                    : {len(df_actives_filtered_qed)}")
print(f"Removed by QED filter                          : {pre_qed_count - len(df_actives_filtered_qed)}")


In [ ]:
df_actives_filtered_qed

In [ ]:
from rdkit.Chem import PandasTools
from rdkit.Chem.FilterCatalog import *
from rdkit.Chem import PandasTools

In [ ]:
#filter by pains
# --- 1. Load PAINS Patterns ---
# Load the catalog containing the widely accepted PAINS filters
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalog = FilterCatalog(params)

In [ ]:
# Define the output file path for the clean actives
output_clean_PAINS_filtered_path = 'outputs/screen_data_predicted_actives_PAINS_filtered.csv'

# --- 2. Define Filtering Function ---
def check_for_pains(mol):
    """Returns True if a PAINS substructure is found, False otherwise."""
    # The FilterCatalog returns all matches; we only care if a match exists.
    if catalog.GetMatches(mol):
        return True
    return False

# --- 3. Prepare Data and Apply Filter ---
print(f"Starting PAINS filtering on {df_actives_filtered_qed.shape[0]} predicted active compounds...")

# Apply the filter
# NOTE: It's good practice to filter for non-None Mol objects before filtering
df_actives_filtered_qed['is_PAINS'] = df_actives_filtered_qed['Molecule'].apply(
    lambda x: check_for_pains(x) if x is not None else False
)

# --- 4. Final Filtering and Saving ---
screen_data_predicted_actives_PAINS_filtered_df = df_actives_filtered_qed[df_actives_filtered_qed['is_PAINS'] == False]

# Optional: View the compounds flagged as PAINS
flagged_pains_df = df_actives_filtered_qed[df_actives_filtered_qed['is_PAINS'] == True]

print("\n--- PAINS Filtering Results ---")
print(f"Total Predicted Actives (pre-filter): {df_actives_filtered_qed.shape[0]}")
print(f"Compounds Flagged as PAINS: {len(flagged_pains_df)}")
print(f"High-Quality Leads (post-PAINS filter): {len(screen_data_predicted_actives_PAINS_filtered_df)}")

# predicted_clean_df.to_csv(output_clean_PAINS_filtered_path, index=False)
# print(f"Clean results saved as: {screen_data_predicted_actives_PAINS_filtered.csv}")

In [ ]:
screen_data_predicted_actives_PAINS_filtered_df.to_csv(output_clean_PAINS_filtered_path, index=False)
print(f"Clean results saved to: {output_clean_PAINS_filtered_path}")

In [ ]:
df_PAINS_filtered = pd.read_csv('outputs/screen_data_predicted_actives_PAINS_filtered.csv')

In [ ]:
df_PAINS_filtered

In [ ]:
Final_leads_for_docking = df_PAINS_filtered.copy()
Final_leads_for_docking

In [ ]:
Final_leads_for_docking['MolWt'].hist(figsize=(9, 6), grid=False)

In [ ]:
Final_leads_for_docking['LogP'].hist(figsize=(9, 6), grid=False)

In [ ]:
Final_leads_for_docking['HbondDonors'].hist(figsize=(9, 6), grid=False)

In [ ]:
Final_leads_for_docking['HbondAcceptors'].hist(figsize=(9, 6), grid=False)